# Class Weight Analysis

Compute class weights for hemorrhage subtypes, skull fracture, and binary midline shift.

## 1. Setup

In [1]:
from pathlib import Path
from typing import Final

import pandas as pd

In [2]:
METADATA_PATH: Final = Path("../data/train/annotated.csv")

MLS_THRESHOLD_MM: Final = 3.0

FRACTURE_COLUMN: Final = "fracture_prob"
MLS_COLUMN: Final = "MLS_mm"
AREA_SUFFIX: Final = "_Area"

## 2. Load Metadata

In [3]:
metadata = pd.read_csv(METADATA_PATH)

required_columns = {
    FRACTURE_COLUMN,
    MLS_COLUMN,
}

missing = required_columns - set(metadata.columns)
if missing:
    raise ValueError(f"Missing required metadata columns: {sorted(missing)}")

ich_area_columns = [
    column for column in metadata.columns if column.endswith(AREA_SUFFIX)
]

if not ich_area_columns:
    raise ValueError(
        f"No hemorrhage area columns ending with {AREA_SUFFIX!r} were found."
    )

print(f"Slices: {len(metadata):,}")
print(f"Hemorrhage area columns: {ich_area_columns}")

metadata.head()

Slices: 4,959
Hemorrhage area columns: ['IntraventricularHemorrhage_Area', 'IntraparenchymalHemorrhage_Area', 'SubduralHemorrhage_Area', 'EpiduralHemorrhage_Area', 'SubarachnoidHemorrhage_Area']


,series_id,patient_id,slice_id,pixel_spacing_y,pixel_spacing_x,slice_thickness,relative_path,IntraventricularHemorrhage,IntraparenchymalHemorrhage,SubduralHemorrhage,EpiduralHemorrhage,SubarachnoidHemorrhage,IntraventricularHemorrhage_Area,IntraparenchymalHemorrhage_Area,SubduralHemorrhage_Area,EpiduralHemorrhage_Area,SubarachnoidHemorrhage_Area,fracture_prob,MLS_mm,triage_class
0,2265,765202,1.2.392.200036.9116.2.6.1.48.1211476691.145931...,0.468,0.468,8.0,2265/1.2.392.200036.9116.2.6.1.48.1211476691.1...,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.100000,2
1,2265,765202,1.2.392.200036.9116.2.6.1.48.1211476691.145931...,0.468,0.468,8.0,2265/1.2.392.200036.9116.2.6.1.48.1211476691.1...,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.100000,2
2,2265,765202,1.2.392.200036.9116.2.6.1.48.1211476691.145931...,0.468,0.468,8.0,2265/1.2.392.200036.9116.2.6.1.48.1211476691.1...,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.100000,2
3,2265,765202,1.2.392.200036.9116.2.6.1.48.1211476691.145931...,0.468,0.468,8.0,2265/1.2.392.200036.9116.2.6.1.48.1211476691.1...,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.100000,2
4,2265,765202,1.2.392.200036.9116.2.6.1.48.1211476691.145931...,0.468,0.468,8.0,2265/1.2.392.200036.9116.2.6.1.48.1211476691.1...,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,1.776818,2


## 3. Weighting Utilities

In [4]:
def multilabel_weights(indicators: pd.DataFrame) -> pd.DataFrame:
    """Compute inverse-frequency weights for multilabel indicators."""
    counts = indicators.sum().astype(int)

    if counts.eq(0).any():
        empty = counts.index[counts.eq(0)].tolist()
        raise ValueError(f"Classes with no positive samples: {empty}")

    n_samples, n_classes = indicators.shape

    return (
        pd.DataFrame(
            {
                "count": counts,
                "prevalence": counts / n_samples,
                "weight": n_samples / (n_classes * counts),
            }
        )
        .rename_axis("class")
        .sort_values("weight", ascending=False)
    )

In [5]:
def binary_weights(
    target: pd.Series,
    *,
    negative_label: str,
    positive_label: str,
) -> pd.DataFrame:
    """Compute balanced weights for a binary target."""
    if target.isna().any():
        raise ValueError(f"Target {target.name!r} contains missing values.")

    target = target.astype(bool)
    counts = target.value_counts().reindex([False, True], fill_value=0)

    if counts.eq(0).any():
        missing = counts.index[counts.eq(0)].tolist()
        raise ValueError(f"Target {target.name!r} is missing classes: {missing}")

    n_samples = len(target)

    return pd.DataFrame(
        {
            "count": counts.to_numpy(),
            "prevalence": (counts / n_samples).to_numpy(),
            "weight": (n_samples / (2 * counts)).to_numpy(),
        },
        index=[negative_label, positive_label],
    ).rename_axis("class")

In [6]:
def show_weights(table: pd.DataFrame) -> None:
    """Display a formatted weight table."""
    display(table.style.format({"prevalence": "{:.2%}", "weight": "{:.4f}"}))

## 4. Hemorrhage Weights

In [7]:
ich_indicators = metadata[ich_area_columns].fillna(0).gt(0).astype("int8")

ich_indicators.columns = [
    column.removesuffix(AREA_SUFFIX) for column in ich_indicators.columns
]


ich_weight_table = multilabel_weights(ich_indicators)
show_weights(ich_weight_table)

,count,prevalence,weight
class,,,
EpiduralHemorrhage,150,3.02%,6.6120
SubarachnoidHemorrhage,193,3.89%,5.1389
IntraventricularHemorrhage,402,8.11%,2.4672
SubduralHemorrhage,428,8.63%,2.3173
IntraparenchymalHemorrhage,920,18.55%,1.0780


## 5. Skull Fracture Weights

In [8]:
fracture_weight_table = binary_weights(
    metadata[FRACTURE_COLUMN],
    negative_label="No fracture",
    positive_label="Fracture",
)

show_weights(fracture_weight_table)

,count,prevalence,weight
class,,,
No fracture,4699,94.76%,0.5277
Fracture,260,5.24%,9.5365


## 6. Midline Shift Weights

In [9]:
mls_positive = metadata[MLS_COLUMN].gt(MLS_THRESHOLD_MM)
mls_positive.name = "midline_shift_positive"

mls_weight_table = binary_weights(
    mls_positive,
    negative_label=f"MLS <= {MLS_THRESHOLD_MM:.1f} mm",
    positive_label=f"MLS > {MLS_THRESHOLD_MM:.1f} mm",
)

show_weights(mls_weight_table)

,count,prevalence,weight
class,,,
MLS <= 3.0 mm,4052,81.71%,0.6119
MLS > 3.0 mm,907,18.29%,2.7337
